In [1]:
# ============================================
# Pig Posture Recognition
# Single split (image_group) + EMA + weak MixUp/CutMix
# + conditional TTA: flip+swap + pad_ratio=0.30 (only for uncertain)
# (K-fold ensemble REMOVED)
# ============================================

import os, gc, glob, math, time, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import torchvision

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import f1_score

import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

In [2]:
# =========================
# Config
# =========================
SEED = 355

EPOCHS = 15
IMG_SIZE = 384
BATCH_SIZE = 16
NUM_WORKERS = 4

PAD_RATIO = 0.20

PAD_RATIO_TTA_LIST = [0.28, 0.36]
LOG_TTA_STATS = True

LR = 2.5e-4
WD = 0.02
PATIENCE = 3
MIN_DELTA = 5e-4

USE_AMP = True
PRETRAINED = True

LABEL_SMOOTH = 0.02

# weak mix settings
MIX_PROB = 0.5
MIXUP_ALPHA = 0.2
CUTMIX_ALPHA = 0.2

# EMA
EMA_DECAY = 0.999

# Uncertainty thresholds
UNCERT_MAXPROB = 0.78
UNCERT_MARGIN  = 0.12

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

PAD_COLOR = (
    int(IMAGENET_MEAN[0] * 255),
    int(IMAGENET_MEAN[1] * 255),
    int(IMAGENET_MEAN[2] * 255),
)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)

DEVICE: cuda


In [3]:
# =========================
# Data paths
# =========================
def find_train_csv():
    cand = glob.glob("/kaggle/input/**/train.csv", recursive=True)
    if len(cand) == 0:
        raise FileNotFoundError("Could not find train.csv under /kaggle/input")
    cand = sorted(cand, key=lambda p: ("pig" not in p.lower(), len(p)))
    return cand[0]

TRAIN_CSV = find_train_csv()
DATA_ROOT = os.path.dirname(TRAIN_CSV)
TEST_CSV  = os.path.join(DATA_ROOT, "test.csv")

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("DATA_ROOT:", DATA_ROOT)
print("train:", train_df.shape, "test:", test_df.shape)
print("train cols:", list(train_df.columns))

DATA_ROOT: /kaggle/input/pig-posture-recognition/pig_posture_recognition
train: (16062, 6) test: (6872, 5)
train cols: ['row_id', 'image_id', 'width', 'height', 'bbox', 'class_id']


In [4]:
def pick_col(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

cols = set(train_df.columns)
ROW_ID_COL = pick_col(cols, ["row_id", "id", "instance_id"])
LABEL_COL  = pick_col(cols, ["class_id", "label", "target"])
IMG_COL    = pick_col(cols, ["image_id", "image", "filename", "file_name"])
BBOX_COL   = pick_col(cols, ["bbox", "box", "bboxes", "boxes"])

XMIN_COL = pick_col(cols, ["xmin", "x_min", "x1", "left"])
YMIN_COL = pick_col(cols, ["ymin", "y_min", "y1", "top"])
W_COL    = pick_col(cols, ["w", "width", "bbox_w"])
H_COL    = pick_col(cols, ["h", "height", "bbox_h"])
XMAX_COL = pick_col(cols, ["xmax", "x_max", "x2", "right"])
YMAX_COL = pick_col(cols, ["ymax", "y_max", "y2", "bottom"])

assert ROW_ID_COL and LABEL_COL and IMG_COL, "Missing essential columns"
print("Using:", ROW_ID_COL, LABEL_COL, IMG_COL, "BBOX_COL:", BBOX_COL)

TRAIN_IMG_DIR = os.path.join(DATA_ROOT, "train_images")
TEST_IMG_DIR  = os.path.join(DATA_ROOT, "test_images")
if not os.path.isdir(TRAIN_IMG_DIR) or not os.path.isdir(TEST_IMG_DIR):
    print("[WARN] train_images/test_images not found; fallback search will be used.")

def resolve_path(img_dir, image_name):
    p = os.path.join(img_dir, str(image_name))
    if os.path.isfile(p): 
        return p
    base = os.path.basename(str(image_name))
    g = glob.glob(os.path.join(DATA_ROOT, f"**/{base}"), recursive=True)
    if len(g) > 0: 
        return g[0]
    raise FileNotFoundError(f"Image not found: {image_name}")

def parse_bbox(row):
    if BBOX_COL is not None and BBOX_COL in row.index:
        s = str(row[BBOX_COL]).strip().strip("[]")
        parts = [p.strip() for p in s.split(",") if p.strip() != ""]
        if len(parts) == 4:
            x, y, w, h = [float(v) for v in parts]
            return x, y, x+w, y+h
    if (XMIN_COL and YMIN_COL and XMAX_COL and YMAX_COL):
        return float(row[XMIN_COL]), float(row[YMIN_COL]), float(row[XMAX_COL]), float(row[YMAX_COL])
    if (XMIN_COL and YMIN_COL and W_COL and H_COL):
        x1, y1 = float(row[XMIN_COL]), float(row[YMIN_COL])
        return x1, y1, x1+float(row[W_COL]), y1+float(row[H_COL])
    raise ValueError("No supported bbox format found.")

def image_group_id(x):
    return os.path.splitext(str(x))[0]

train_df["image_group"] = train_df[IMG_COL].map(image_group_id)
test_df["image_group"]  = test_df[IMG_COL].map(image_group_id)
train_df[LABEL_COL] = train_df[LABEL_COL].astype(int)

Using: row_id class_id image_id BBOX_COL: bbox


In [5]:
# =========================
# Augmentations
# =========================
train_aug = A.Compose([
    A.Affine(scale=(0.9, 1.1), translate_percent=(0.0, 0.08), rotate=(-10, 10), p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.02, p=0.5),
    A.CoarseDropout(num_holes_range=(1, 6), hole_height_range=(16, 48), hole_width_range=(16, 48), p=0.15),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

val_aug = A.Compose([
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

tta_norm = val_aug

# =========================
# Dataset
# =========================
class PigCropDataset(Dataset):
    def __init__(self, df, img_dir, transforms=None, train=True, pad_ratio=0.20):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transforms = transforms
        self.train = train
        self.pad_ratio = float(pad_ratio)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        path = resolve_path(self.img_dir, row[IMG_COL])
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        H, W, _ = img.shape

        x1, y1, x2, y2 = parse_bbox(row)

        # context padding around bbox
        bw, bh = x2 - x1, y2 - y1
        pad = self.pad_ratio * max(bw, bh)

        x1 = max(0, int(x1 - pad))
        y1 = max(0, int(y1 - pad))
        x2 = min(W, int(x2 + pad))
        y2 = min(H, int(y2 + pad))

        crop = img[y1:y2, x1:x2]

        # letterbox to square with ImageNet mean padding
        h_crop, w_crop, _ = crop.shape
        max_side = max(h_crop, w_crop)

        square_img = np.empty((max_side, max_side, 3), dtype=np.uint8)
        square_img[:] = PAD_COLOR

        y_off = (max_side - h_crop) // 2
        x_off = (max_side - w_crop) // 2
        square_img[y_off:y_off+h_crop, x_off:x_off+w_crop] = crop

        final_img = cv2.resize(square_img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)

        if self.transforms:
            final_img = self.transforms(image=final_img)["image"]

        if self.train:
            return final_img, int(row[LABEL_COL])
        else:
            return final_img, str(row[ROW_ID_COL])

In [6]:
# =========================
# Model
# =========================
def build_model(num_classes=5):
    try:
        weights = torchvision.models.ConvNeXt_Small_Weights.IMAGENET1K_V1 if PRETRAINED else None
        m = torchvision.models.convnext_small(weights=weights)
    except Exception as e:
        print("[WARN] Pretrained weights not available -> random init. Error:", e)
        m = torchvision.models.convnext_small(weights=None)

    in_features = m.classifier[2].in_features
    m.classifier[2] = nn.Linear(in_features, num_classes)
    return m

In [7]:
# =========================
# Metrics
# =========================
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type="cuda", enabled=USE_AMP and DEVICE == "cuda"):
            pred = model(x).argmax(1).detach().cpu().numpy()
        ys.append(y.numpy())
        ps.append(pred)
    return f1_score(np.concatenate(ys), np.concatenate(ps), average="macro")

In [8]:
# =========================
# EMA
# =========================
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = float(decay)
        self.shadow = {}
        self.backup = {}
        for name, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[name] = p.detach().clone()

    @torch.no_grad()
    def update(self, model):
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            new = p.detach()
            old = self.shadow[name]
            self.shadow[name] = old * self.decay + new * (1.0 - self.decay)

    def apply_shadow(self, model):
        self.backup = {}
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            self.backup[name] = p.detach().clone()
            p.data.copy_(self.shadow[name].data)

    def restore(self, model):
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            p.data.copy_(self.backup[name].data)
        self.backup = {}


In [9]:
# =========================
# MixUp / CutMix helpers
# =========================
def rand_bbox(size, lam):
    # size: (B, C, H, W)
    W = size[3]
    H = size[2]
    cut_rat = np.sqrt(1.0 - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    cx = np.random.randint(W)
    cy = np.random.randint(H)

    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    return x1, y1, x2, y2

def soft_ce_loss(logits, y_a, y_b, lam):
    return lam * F.cross_entropy(logits, y_a, label_smoothing=LABEL_SMOOTH) + \
           (1.0 - lam) * F.cross_entropy(logits, y_b, label_smoothing=LABEL_SMOOTH)


In [10]:
# =========================
# TTA helpers (flip + swap)
# =========================
# class mapping (your assumed order)
LEFT_ID  = 0
RIGHT_ID = 1

def swap_left_right_probs(p):
    p2 = p.clone()
    p2[:, LEFT_ID]  = p[:, RIGHT_ID]
    p2[:, RIGHT_ID] = p[:, LEFT_ID]
    return p2

@torch.no_grad()
def probs(model, x):
    with torch.amp.autocast(device_type="cuda", enabled=USE_AMP and DEVICE == "cuda"):
        return torch.softmax(model(x), dim=1)

def is_uncertain(p):
    top2 = torch.topk(p, k=2, dim=1).values
    maxp = top2[:, 0]
    margin = top2[:, 0] - top2[:, 1]
    return (maxp < UNCERT_MAXPROB) | (margin < UNCERT_MARGIN)

In [11]:
# =========================
# Single split by image_group (dominant label stratified)
# =========================
img_dom = (
    train_df.groupby("image_group")[LABEL_COL]
    .agg(lambda s: int(s.value_counts().idxmax()))
    .reset_index()
    .rename(columns={LABEL_COL: "dom_label"})
)

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.08, random_state=SEED)
tr_i, va_i = next(sss.split(img_dom["image_group"], img_dom["dom_label"]))

tr_imgs = set(img_dom.iloc[tr_i]["image_group"].tolist())
va_imgs = set(img_dom.iloc[va_i]["image_group"].tolist())

trn = train_df[train_df["image_group"].isin(tr_imgs)].reset_index(drop=True)
val = train_df[train_df["image_group"].isin(va_imgs)].reset_index(drop=True)

print("Train instances:", len(trn), "Val instances:", len(val))

num_classes = int(train_df[LABEL_COL].max()) + 1
print("num_classes:", num_classes)

Train instances: 14813 Val instances: 1249
num_classes: 5


In [12]:
# =========================
# Sampler
# =========================
class_counts = trn[LABEL_COL].value_counts().sort_index().values.astype(np.float32)
class_weights = 1.0 / np.clip(class_counts, 1.0, None)
sample_weights = trn[LABEL_COL].map(lambda c: class_weights[int(c)]).values

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# loaders
ds_tr = PigCropDataset(trn, TRAIN_IMG_DIR, transforms=train_aug, train=True,  pad_ratio=PAD_RATIO)
ds_va = PigCropDataset(val, TRAIN_IMG_DIR, transforms=val_aug,   train=True,  pad_ratio=PAD_RATIO)

dl_tr = DataLoader(
    ds_tr, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
    persistent_workers=True, prefetch_factor=2
)
dl_va = DataLoader(
    ds_va, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True, prefetch_factor=2
)

In [13]:
# =========================
# Train setup
# =========================
model = build_model(num_classes=num_classes).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.05)

scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and DEVICE == "cuda")
ema = EMA(model, decay=EMA_DECAY)

best_f1, bad = -1.0, 0
best_path = "/kaggle/working/best_convnext.pt"

for ep in range(EPOCHS):
    t0 = time.time()
    model.train()
    running, seen = 0.0, 0

    for x, y in dl_tr:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        use_mix = (random.random() < MIX_PROB)

        if use_mix:
            r = random.random()
            if r < 0.5:
                # MixUp
                lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
                idx = torch.randperm(x.size(0), device=x.device)
                x2 = x[idx]
                y_a, y_b = y, y[idx]
                x_mix = lam * x + (1.0 - lam) * x2

                with torch.amp.autocast(device_type="cuda", enabled=USE_AMP and DEVICE == "cuda"):
                    logits = model(x_mix)
                    loss = soft_ce_loss(logits, y_a, y_b, lam)
            else:
                # CutMix
                lam = np.random.beta(CUTMIX_ALPHA, CUTMIX_ALPHA)
                idx = torch.randperm(x.size(0), device=x.device)
                y_a, y_b = y, y[idx]

                bbx1, bby1, bbx2, bby2 = rand_bbox(x.size(), lam)
                x_cm = x.clone()
                x_cm[:, :, bby1:bby2, bbx1:bbx2] = x[idx, :, bby1:bby2, bbx1:bbx2]

                area = (bbx2 - bbx1) * (bby2 - bby1)
                lam = 1.0 - area / (x.size(2) * x.size(3))

                with torch.amp.autocast(device_type="cuda", enabled=USE_AMP and DEVICE == "cuda"):
                    logits = model(x_cm)
                    loss = soft_ce_loss(logits, y_a, y_b, lam)
        else:
            with torch.amp.autocast(device_type="cuda", enabled=USE_AMP and DEVICE == "cuda"):
                loss = criterion(model(x), y)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        ema.update(model)

        running += float(loss.item()) * x.size(0)
        seen += x.size(0)

    scheduler.step()

    # EMA evaluation
    ema.apply_shadow(model)
    val_f1 = evaluate(model, dl_va)
    ema.restore(model)

    dt = (time.time() - t0) / 60.0
    print(f"Epoch {ep+1:02d}/{EPOCHS} | loss={running/max(1,seen):.4f} | val_f1={val_f1:.5f} | {dt:.2f} min")

    if val_f1 > best_f1 + MIN_DELTA:
        best_f1 = val_f1
        bad = 0
        ema.apply_shadow(model)
        torch.save({"model": model.state_dict()}, best_path)
        ema.restore(model)
        print("  -> saved best:", best_path)
    else:
        bad += 1
        if bad >= PATIENCE:
            print("Early stopping.")
            break

print("Best val macro-F1:", best_f1)

Downloading: "https://download.pytorch.org/models/convnext_small-0c510722.pth" to /root/.cache/torch/hub/checkpoints/convnext_small-0c510722.pth


100%|██████████| 192M/192M [00:00<00:00, 219MB/s]


Epoch 01/15 | loss=0.8855 | val_f1=0.73890 | 13.88 min
  -> saved best: /kaggle/working/best_convnext.pt
Epoch 02/15 | loss=0.5309 | val_f1=0.88247 | 13.66 min
  -> saved best: /kaggle/working/best_convnext.pt
Epoch 03/15 | loss=0.4445 | val_f1=0.92831 | 13.66 min
  -> saved best: /kaggle/working/best_convnext.pt
Epoch 04/15 | loss=0.4145 | val_f1=0.94334 | 13.66 min
  -> saved best: /kaggle/working/best_convnext.pt
Epoch 05/15 | loss=0.4332 | val_f1=0.94115 | 13.65 min
Epoch 06/15 | loss=0.4048 | val_f1=0.94810 | 13.66 min
  -> saved best: /kaggle/working/best_convnext.pt
Epoch 07/15 | loss=0.3720 | val_f1=0.95845 | 13.66 min
  -> saved best: /kaggle/working/best_convnext.pt
Epoch 08/15 | loss=0.3568 | val_f1=0.95710 | 13.65 min
Epoch 09/15 | loss=0.3479 | val_f1=0.96417 | 13.66 min
  -> saved best: /kaggle/working/best_convnext.pt
Epoch 10/15 | loss=0.3346 | val_f1=0.96245 | 13.66 min
Epoch 11/15 | loss=0.3065 | val_f1=0.96750 | 13.65 min
  -> saved best: /kaggle/working/best_convnex

In [14]:
# =========================
# Inference + conditional TTA
#   - base pad_ratio=PAD_RATIO
#   - flip+swap only for uncertain
#   - pad_ratio TTA 3 variants only for uncertain (averaged)
#   + [ADDED] Check A: uncertain ratio
#   + [ADDED] Check B: argmax change rate by pad-avg (within uncertain set)
# =========================
ckpt = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(ckpt["model"], strict=True)
model.eval()

test_ds = PigCropDataset(test_df, TEST_IMG_DIR, transforms=tta_norm, train=False, pad_ratio=PAD_RATIO)
test_dl = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

test_ds_pads = [
    PigCropDataset(test_df, TEST_IMG_DIR, transforms=tta_norm, train=False, pad_ratio=r)
    for r in PAD_RATIO_TTA_LIST
]
test_dl_pads = [
    DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    for ds in test_ds_pads
]
pad_iters = [iter(dl) for dl in test_dl_pads]

# ===== [ADDED] stats accumulators =====
total_n = 0

# Check A-1: base pass에서 uncertain 비율
unc_base_n = 0

# (참고로 보면 도움됨) Check A-2: flip 후에도 남은 uncertain(=pad 들어가는 애들) 비율
unc_afterflip_n = 0

# Check B: pad 평균이 argmax를 바꾸는 비율 (pad 들어간 애들 중)
pad_eval_n = 0
pad_argmax_changed_n = 0

all_ids, all_pred = [], []

for (x, rid) in test_dl:
    bs = x.size(0)
    total_n += bs

    x = x.to(DEVICE, non_blocking=True)

    # pad 3종 배치 로드(순서 확인)
    x_pads = []
    rid_ref = list(rid)
    for it in pad_iters:
        x_pad, rid2 = next(it)
        if list(rid2) != rid_ref:
            raise RuntimeError("test_dl and pad TTA loader ordering mismatch. Check shuffle=False and identical df order.")
        x_pads.append(x_pad.to(DEVICE, non_blocking=True))

    # base pass
    p = probs(model, x)

    # ===== [ADDED] Check A-1 =====
    mask0 = is_uncertain(p)                 # base에서 불확실
    unc_base_n += int(mask0.sum().item())

    # conditional flip+swap (on current p)
    if mask0.any():
        x_flip = torch.flip(x[mask0], dims=[3])
        p_flip = probs(model, x_flip)
        p_flip = swap_left_right_probs(p_flip)
        p[mask0] = (p[mask0] + p_flip) / 2.0

    # conditional pad_ratio TTA 3 variants (average only on uncertain)
    mask1 = is_uncertain(p)                 # flip 반영 후에도 불확실 (pad 들어갈 애들)
    unc_afterflip_n += int(mask1.sum().item())

    if mask1.any():
        # ===== [ADDED] Check B 준비: pad 적용 전 argmax 저장 =====
        pred_before_pad = p.argmax(dim=1)

        # pad 3종 평균
        p_sum = 0.0
        for x_pad in x_pads:
            p_sum = p_sum + probs(model, x_pad)[mask1]
        p_avg = p_sum / float(len(x_pads))

        # base/flip 결과와 pad 평균을 한번 더 평균
        p[mask1] = (p[mask1] + p_avg) / 2.0

        # ===== [ADDED] Check B: argmax가 바뀌었는지 =====
        pred_after_pad = p.argmax(dim=1)
        changed = (pred_before_pad[mask1] != pred_after_pad[mask1]).sum().item()
        pad_argmax_changed_n += int(changed)
        pad_eval_n += int(mask1.sum().item())

    all_ids.extend(rid_ref)
    all_pred.append(p.argmax(dim=1).detach().cpu())

all_pred = torch.cat(all_pred).numpy().astype(int)
sub = pd.DataFrame({"row_id": all_ids, "class_id": all_pred})
sub.to_csv("/kaggle/working/submission.csv", index=False)

print(sub.head())
print("Saved:", "/kaggle/working/submission.csv")

# ===== [ADDED] stats print =====
if ('LOG_TTA_STATS' not in globals()) or LOG_TTA_STATS:
    base_unc_ratio = unc_base_n / max(1, total_n)
    afterflip_unc_ratio = unc_afterflip_n / max(1, total_n)

    pad_change_ratio = (pad_argmax_changed_n / max(1, pad_eval_n)) if pad_eval_n > 0 else 0.0

    print("\n[TTA STATS]")
    print(f"Total samples: {total_n}")
    print(f"Check A (base uncertain ratio): {unc_base_n}/{total_n} = {base_unc_ratio*100:.2f}%")
    print(f"Check A (after-flip uncertain ratio -> pad-applied candidates): {unc_afterflip_n}/{total_n} = {afterflip_unc_ratio*100:.2f}%")
    print(f"Check B (pad-avg changed argmax within pad-applied set): {pad_argmax_changed_n}/{pad_eval_n} = {pad_change_ratio*100:.2f}%")


               row_id  class_id
0  test_00002165_0000         3
1  test_00002165_0001         3
2  test_00002165_0002         3
3  test_00002165_0003         3
4  test_00002165_0004         3
Saved: /kaggle/working/submission.csv

[TTA STATS]
Total samples: 6872
Check A (base uncertain ratio): 70/6872 = 1.02%
Check A (after-flip uncertain ratio -> pad-applied candidates): 51/6872 = 0.74%
Check B (pad-avg changed argmax within pad-applied set): 28/51 = 54.90%
